# Reset Azure SQL tables and schemas

This notebook drops **all user tables and their data in the configured database**, including tables in `dbo` and any custom schema, then drops all custom schemas. It is not limited to Bronze, Silver, Gold, and QA. Built-in schemas (`dbo`, `guest`, `sys`, `INFORMATION_SCHEMA`, and database-role schemas) and the database itself remain.

Run the preview first and review the server, database, object list, and generated SQL. The execution cell defaults to disabled. No QuickBooks operations are performed. Stop pipeline runs and other database writers before cleanup.

The project uses `src.azure_sql.get_engine()` and Azure SQL settings from `.env`. Only Azure SQL credentials are required. This utility requires CONTROL permission on the target database. Successful cleanup permanently removes the selected tables; ensure any needed data has been backed up before execution.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "azure_sql.py").exists():
    raise FileNotFoundError("Open from the project root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.azure_sql import get_engine
from src.azure_sql_cleanup import inspect_cleanup, execute_cleanup

required = ("AZURE_SQL_SERVER", "AZURE_SQL_DATABASE", "AZURE_SQL_USERNAME", "AZURE_SQL_PASSWORD")
missing = [name for name in required if not getattr(config, name)]
if missing:
    raise ValueError(f"Missing Azure SQL configuration: {missing}")
engine = get_engine()

## Execution settings

In [ ]:
APPLY_CLEANUP = False
CONFIRM_DATABASE = ""  # Enter the exact database name shown by the preview.

## Preview (read-only)

Foreign keys are removed before tables. For temporal tables, system versioning is disabled before dropping the current and history tables. Other schema objects (such as views, procedures, functions, and user-defined types) are reported as blockers rather than deleted. Ledger, external, memory-optimized, and FileTable objects require separate handling and block execution. Unexpected dependencies cause the transaction to roll back.

In [ ]:
with engine.connect() as connection:
    cleanup_plan = inspect_cleanup(connection)

print("Server:", cleanup_plan["identity"]["server_name"])
print("Database:", cleanup_plan["identity"]["database_name"])
print("User tables to drop:", len(cleanup_plan["tables"]))
print("Custom schemas to drop:", len(cleanup_plan["schemas"]))
display(pd.DataFrame(cleanup_plan["tables"]))
display(pd.DataFrame(cleanup_plan["schemas"]))
if cleanup_plan["blockers"]:
    print("BLOCKED: review these objects before proceeding.")
    display(pd.DataFrame(cleanup_plan["blockers"]))
print("\nSQL preview:\n" + "\n".join(cleanup_plan["statements"]))

## Execute the reviewed cleanup

Set `APPLY_CLEANUP = True` and fill `CONFIRM_DATABASE` in the settings cell, then rerun that cell and this one. The helper rechecks the live target and metadata against the preview. It executes the DDL in one transaction and verifies that no user tables or custom schemas remain before committing. Any error rolls back the transaction; review the error and rerun the preview before retrying.

In [ ]:
if not APPLY_CLEANUP:
    print("Preview only. No tables or schemas were dropped.")
else:
    cleanup_result = execute_cleanup(engine, cleanup_plan, CONFIRM_DATABASE)
    print("Cleanup committed:", cleanup_result)

## Release connections and rebuild

After cleanup, the extraction and transformation notebooks can recreate the pipeline schemas and tables. `99_run_pipeline.ipynb` provides the full analytics refresh. Cleanup is a separate maintenance action and is never called automatically by that pipeline.

References: [DROP TABLE](https://learn.microsoft.com/en-us/sql/t-sql/statements/drop-table-transact-sql) and [DROP SCHEMA](https://learn.microsoft.com/en-us/sql/t-sql/statements/drop-schema-transact-sql).

In [ ]:
engine.dispose()